# 4. Performance Tracking

## Overview
This notebook implements performance tracking for our ETF portfolio, including:
1. Historical price data collection using Alpha Vantage
2. Return calculations (monthly and annual)
3. Risk metrics calculation
4. Performance reporting

## Required Libraries

In [1]:
import pandas as pd
import numpy as np
import requests

# Load our portfolio
portfolio_df = pd.read_csv('final_portfolio.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'final_portfolio.csv'

## ETF Performance Calculator
Using your existing code to calculate ETF metrics:

In [2]:
def calculate_etf_metrics(symbol, custom_start_end=None):
    """Calculate annual and monthly metrics for an ETF."""
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
    
    r = requests.get(url)
    data = r.json()
    df = pd.DataFrame(data['Time Series (Daily)']).T
    
    # Convert to pounds and round
    df['5. adjusted close'] = pd.to_numeric(df['5. adjusted close'], errors='coerce') / 100
    df['5. adjusted close'] = df['5. adjusted close'].round(2)
    
    df.index = pd.to_datetime(df.index)
    
    return calculate_annual_metrics(df, custom_start_end)

## Annual Performance Metrics

In [3]:
def calculate_annual_metrics(df, custom_start_end=None):
    """Calculate annual performance metrics."""
    annual_results = []
    
    for year in range(2021, 2025):
        df_year = df[df.index.year == year]
        if df_year.empty:
            continue
            
        # Get start/end prices
        if custom_start_end and str(year) in custom_start_end:
            start_price = custom_start_end[str(year)].get("start_price", df_year['5. adjusted close'].iloc[-1])
            end_price = custom_start_end[str(year)].get("end_price", df_year['5. adjusted close'].iloc[0])
        else:
            start_price = df_year['5. adjusted close'].iloc[-1]
            end_price = df_year['5. adjusted close'].iloc[0]
            
        # Calculate returns and volatility
        annual_return = ((end_price / start_price) - 1)
        df_year['daily_return'] = df_year['5. adjusted close'].pct_change()
        volatility = df_year['daily_return'].std() * np.sqrt(252)
        
        annual_results.append({
            "Year": year,
            "Start Price": round(start_price, 2),
            "End Price": round(end_price, 2),
            "Annualized Return (%)": round(annual_return * 100, 2),
            "Volatility (%)": round(volatility * 100, 2)
        })
    
    return pd.DataFrame(annual_results)

## Portfolio Performance
Calculate weighted portfolio performance:

## Calculate Portfolio Performance
First, we'll calculate the performance metrics for each ETF and combine them into portfolio-level metrics:

In [4]:
# Initialize portfolio performance tracking
portfolio_performance = pd.DataFrame()

# Calculate performance for each ETF
for idx, row in portfolio_df.iterrows():
    try:
        # Get ETF metrics 
        etf_metrics = calculate_etf_metrics(row['ticker']+".LON")
        if not etf_metrics.empty:
            # Weight the metrics by investment amount
            weight = row['investment'] / portfolio_df['investment'].sum()
            etf_metrics['Weighted Return (%)'] = etf_metrics['Annualized Return (%)'] * weight
            etf_metrics['Weighted Volatility (%)'] = etf_metrics['Volatility (%)'] * weight
            etf_metrics['ETF'] = row['ticker']
            portfolio_performance = pd.concat([portfolio_performance, etf_metrics])
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

# Calculate portfolio-level metrics
if not portfolio_performance.empty:
    portfolio_summary = pd.DataFrame({
        'Year': portfolio_performance['Year'].unique(),
        'Return (%)': portfolio_performance.groupby('Year')['Weighted Return (%)'].sum().values,
        'Volatility (%)': portfolio_performance.groupby('Year')['Weighted Volatility (%)'].sum().values
    })
    
    print("\nPortfolio Performance Summary:")
    print(portfolio_summary)
else:
    print("No performance data available")

C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year['daily_return'] = df_year['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year['daily_return'] = df_year['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a


Portfolio Performance Summary:
   Year  Return (%)  Volatility (%)
0  2021      4.3154         19.7991
1  2022     -1.9076         29.1223
2  2023     14.8612         20.6568
3  2024      5.7722         16.8860


C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year['daily_return'] = df_year['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_year['daily_return'] = df_year['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\4156835603.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a

In [5]:
# Now calculate risk metrics using the portfolio summary
if 'portfolio_summary' in locals() and not portfolio_summary.empty:
    risk_metrics = calculate_risk_metrics(portfolio_summary)
    print("\nRisk-Adjusted Performance Metrics:")
    print(risk_metrics)
else:
    print("Unable to calculate risk metrics - no performance data available")

NameError: name 'calculate_risk_metrics' is not defined

## Recent Performance Metrics (YTD and MTD)
Calculate performance metrics for:
1. Year to Date (2025)
2. Month to Date (April 2025)

In [6]:
def calculate_period_metrics(df, start_date, end_date=None):
    """Calculate performance metrics for a specific time period."""
    if end_date is None:
        end_date = pd.Timestamp.now()
    
    # Filter data for the period
    mask = (df.index >= start_date) & (df.index <= end_date)
    period_data = df[mask]
    
    if period_data.empty:
        return None
    
    # Calculate returns and volatility
    start_price = period_data['5. adjusted close'].iloc[-1]
    end_price = period_data['5. adjusted close'].iloc[0]
    period_return = ((end_price / start_price) - 1) * 100
    
    # Calculate annualized volatility
    period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
    volatility = period_data['daily_return'].std() * np.sqrt(252) * 100
    
    return {
        'Start Date': start_date.strftime('%Y-%m-%d'),
        'End Date': end_date.strftime('%Y-%m-%d'),
        'Return (%)': round(period_return, 2),
        'Volatility (%)': round(volatility, 2)
    }

In [7]:
# Calculate YTD and MTD performance
ytd_start = pd.Timestamp('2025-01-01')
mtd_start = pd.Timestamp('2025-05-12')  # Current date specified in context
current_date = pd.Timestamp('2025-05-14')

ytd_performance = []
mtd_performance = []

for idx, row in portfolio_df.iterrows():
    try:
        # Get ETF data
        url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={row["ticker"]+".lon"}&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
        r = requests.get(url)
        data = r.json()
        
        if 'Time Series (Daily)' in data:
            df = pd.DataFrame(data['Time Series (Daily)']).T
            df.index = pd.to_datetime(df.index)
            df['5. adjusted close'] = pd.to_numeric(df['5. adjusted close'], errors='coerce')
            
            # Calculate metrics
            weight = row['investment'] / portfolio_df['investment'].sum()
            
            # YTD metrics
            ytd_metrics = calculate_period_metrics(df, ytd_start, current_date)
            if ytd_metrics:
                ytd_metrics['ETF'] = row['ticker']
                ytd_metrics['Weight'] = weight
                ytd_metrics['Weighted Return (%)'] = ytd_metrics['Return (%)'] * weight
                ytd_metrics['Weighted Volatility (%)'] = ytd_metrics['Volatility (%)'] * weight
                ytd_performance.append(ytd_metrics)
            
            # MTD metrics
            mtd_metrics = calculate_period_metrics(df, mtd_start, current_date)
            if mtd_metrics:
                mtd_metrics['ETF'] = row['ticker']
                mtd_metrics['Weight'] = weight
                mtd_metrics['Weighted Return (%)'] = mtd_metrics['Return (%)'] * weight
                mtd_metrics['Weighted Volatility (%)'] = mtd_metrics['Volatility (%)'] * weight
                mtd_performance.append(mtd_metrics)
                
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

# Convert to DataFrames
ytd_df = pd.DataFrame(ytd_performance)
mtd_df = pd.DataFrame(mtd_performance)

# Calculate portfolio level metrics
if not ytd_df.empty:
    print("\nYear to Date (2025) Portfolio Performance:")
    print(f"Total Return: {ytd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {ytd_df['Weighted Volatility (%)'].sum():.2f}%")
    print("\nTop 3 YTD Contributors:")
    print(ytd_df.nlargest(3, 'Weighted Return (%)')[['ETF', 'Return (%)', 'Weighted Return (%)']])

if not mtd_df.empty:
    print("\nMonth to Date (April 2025) Portfolio Performance:")
    print(f"Total Return: {mtd_df['Weighted Return (%)'].sum():.2f}%")
    print(f"Portfolio Volatility: {mtd_df['Weighted Volatility (%)'].sum():.2f}%")
    print("\nTop 3 MTD Contributors:")
    print(mtd_df.nlargest(3, 'Weighted Return (%)')[['ETF', 'Return (%)', 'Weighted Return (%)']]),

# Calculate Sharpe ratios for recent periods
risk_free_rate = 0.02  # Assuming 2% risk-free rate
if not ytd_df.empty:
    ytd_sharpe = ((ytd_df['Weighted Return (%)'].sum() / 100) - (risk_free_rate * (116/252))) / (ytd_df['Weighted Volatility (%)'].sum() / 100)
    print(f"\nYTD Sharpe Ratio: {ytd_sharpe:.2f}")

if not mtd_df.empty:
    mtd_sharpe = ((mtd_df['Weighted Return (%)'].sum() / 100) - (risk_free_rate * (1/252))) / (mtd_df['Weighted Volatility (%)'].sum() / 100)
    print(f"MTD Sharpe Ratio: {mtd_sharpe:.2f}")

C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\1586149106.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\1586149106.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\1586149106.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy o


Year to Date (2025) Portfolio Performance:
Total Return: 7.67%
Portfolio Volatility: 20.29%

Top 3 YTD Contributors:
    ETF  Return (%)  Weighted Return (%)
4  IMIB       20.75               3.9425
5  XDDX       16.97               2.8849
7  IBZL       16.72               1.5048

Month to Date (April 2025) Portfolio Performance:
Total Return: 0.23%
Portfolio Volatility: 7.78%

Top 3 MTD Contributors:
    ETF  Return (%)  Weighted Return (%)
7  IBZL        3.02               0.2718
4  IMIB        1.24               0.2356
6  HMCH        0.92               0.0184

YTD Sharpe Ratio: 0.33
MTD Sharpe Ratio: 0.03


C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\1586149106.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_33416\1586149106.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()


In [28]:
#calculate and print day by day  portfolio pnl 
ytd_start = pd.Timestamp('2025-05-12')
def calculate_daily_pnl(df, investment, start_date, end_date=None):
    """Calculate daily PnL for a specific time period."""
    if end_date is None:
        end_date = pd.Timestamp.now()
    
    # Filter data for the period
    mask = (df.index >= start_date) & (df.index <= end_date)
    period_data = df[mask]
    
    if period_data.empty:
        return None
    
    # Calculate daily PnL
    period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
    period_data['PnL'] = period_data['daily_return'] * investment
    
    return period_data[['PnL']]
# Calculate daily PnL for the portfolio
daily_pnl = pd.DataFrame()
for idx, row in portfolio_df.iterrows():
    try:
        # Get ETF data
        url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={row["ticker"]+".lon"}&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
        r = requests.get(url)
        data = r.json()
        
        if 'Time Series (Daily)' in data:
            df = pd.DataFrame(data['Time Series (Daily)']).T
            df.index = pd.to_datetime(df.index)
            df['5. adjusted close'] = pd.to_numeric(df['5. adjusted close'], errors='coerce')
            
            # Calculate daily PnL
            daily_pnl_etf = calculate_daily_pnl(df,row['investment'], ytd_start, current_date)
            if daily_pnl_etf is not None:
                print (f"Calculating daily PnL for {row['ticker']}")
                daily_pnl_etf['ETF'] = row['ticker']
                print(row['investment'])
                daily_pnl_etf['Weight'] = row['investment'] / portfolio_df['investment'].sum()
                daily_pnl_etf['Weighted PnL'] = daily_pnl_etf['PnL'] * daily_pnl_etf['Weight']
                daily_pnl = pd.concat([daily_pnl, daily_pnl_etf])
                
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")
# Print daily PnL
if not daily_pnl.empty:
    print("\nDaily PnL for the Portfolio:")
    print(daily_pnl.groupby('ETF')['Weighted PnL'].sum())   


C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


Calculating daily PnL for AUAD
3200.0


C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


Calculating daily PnL for PRIJ
1400.0


C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


...

C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


Calculating daily PnL for VEMT
200.0


C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


Calculating daily PnL for PRIR
600.0
Calculating daily PnL for VECP
1200.0

Daily PnL for the Portfolio:
ETF
AUAD    0.404051
EMCP    0.013552
HMCH   -0.072804
IBZL   -3.768761
IGLT    0.005025
IMIB   -7.935459
LCUK    0.386751
PRIJ    2.231643
PRIR    0.029849
QYLP    0.661694
SLXX   -0.007451
TRXG    0.022629
UC81    0.013953
VECP   -0.030771
VEMT    0.007201
XDDX    0.483635
Name: Weighted PnL, dtype: float64


C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['daily_return'] = period_data['5. adjusted close'].pct_change()
C:\Users\rakes\AppData\Local\Temp\ipykernel_53664\4081678951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  period_data['PnL'] = period_data['daily_return'] * investment


In [12]:
import polars as pl
from datetime import datetime

ytd_start = datetime(2025, 5, 12).date()
current_date = datetime.now().date()

def calculate_daily_pnl(df: pl.DataFrame, investment: float, start_date: datetime, end_date: datetime = None) -> pl.DataFrame:
    """Calculate daily PnL for a specific time period using Polars."""
    if end_date is None:
        end_date = datetime.now()
    
    # Filter data for the period
    period_data = df.filter(
        (pl.col('date') >= start_date) & 
        (pl.col('date') <= end_date)
    )
    
    if period_data.height == 0:
        return None
    
    # Calculate daily PnL
    period_data = period_data.with_columns([
        pl.col('adjusted close')
          .pct_change()
          .alias('daily_return'),
    ]).with_columns([
        (pl.col('daily_return') * investment).alias('PnL')
    ])
    
    return period_data.select(['date', 'daily_return', 'PnL'])

# read the portfolio CSV file using Polars
portfolio_df = pl.read_csv('final_portfolio.csv')

# Calculate daily PnL for the portfolio
daily_pnl_list = []

for row in portfolio_df.iter_rows(named=True):
    try:
        # Get ETF data
        url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={row["ticker"]}.lon&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
        r = requests.get(url)
        data = r.json()
        
        if 'Time Series (Daily)' in data:
            # Convert JSON to Polars DataFrame
            df = pl.DataFrame([
                {'date': datetime.strptime(date, '%Y-%m-%d').date(), **{k.split('. ')[1]: float(v) for k, v in values.items()}}
                for date, values in data['Time Series (Daily)'].items()
            ])
            
             
            # Calculate daily PnL
            daily_pnl_etf = calculate_daily_pnl(df, row['investment'], ytd_start, current_date)
            
            if daily_pnl_etf is not None:                
                # Add ETF and weight information
                daily_pnl_etf = daily_pnl_etf.with_columns([
                    pl.lit(row['ticker']).alias('ETF'),
                    #include returns column
                    pl.lit(row['investment'] / portfolio_df['investment'].sum()).alias('Weight')
                ]).with_columns([
                    (pl.col('PnL') * pl.col('Weight')).alias('Weighted PnL')
                ])
                
                daily_pnl_list.append(daily_pnl_etf)
                
    except Exception as e:
        print(f"Error processing {row['ticker']}: {str(e)}")

# Combine all ETF results
if daily_pnl_list:
    daily_pnl = pl.concat(daily_pnl_list)
    print("\nDaily PnL for the Portfolio:")
    print(daily_pnl.group_by('ETF').agg(pl.col('Weighted PnL').sum()))

FileNotFoundError: The system cannot find the path specified. (os error 3): final_portfolio.csv

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Filter and sort data for AUAD
auad_data = daily_pnl.filter(pl.col("ETF")=="AUAD").sort('date')

# Create the interactive plot
fig = go.Figure()

# Add the line plot with markers
fig.add_trace(
    go.Scatter(
        x=auad_data.get_column('date'),
        y=auad_data.get_column('daily_return'),
        mode='lines+markers',
        name='Daily Return',
        line=dict(color='blue'),
        hovertemplate='Date: %{x}<br>Return: %{y:.2%}<extra></extra>'
    )
)

# Update layout
fig.update_layout(
    title='AUAD ETF Daily Returns',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    yaxis_tickformat='.2%',
    hovermode='x unified',
    template='plotly_white',
    width=1000,
    height=600
)

# Add gridlines
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Show the plot
fig.show()


In [8]:
auad_data

date,daily_return,PnL,ETF,Weight,Weighted PnL
date,f64,f64,str,f64,f64
2025-05-12,-0.009981,-31.939574,"""AUAD""",0.16,-5.110332
2025-05-13,0.01077,34.464894,"""AUAD""",0.16,5.514383
2025-05-14,-0.009052,-28.965145,"""AUAD""",0.16,-4.634423
2025-05-15,0.002302,7.366283,"""AUAD""",0.16,1.178605
2025-05-16,-0.003374,-10.796221,"""AUAD""",0.16,-1.727395
…,…,…,…,…,…
2025-05-30,-0.006465,-20.686869,"""AUAD""",0.16,-3.309899
2025-06-02,-0.003757,-12.022005,"""AUAD""",0.16,-1.923521
2025-06-03,-0.006532,-20.901093,"""AUAD""",0.16,-3.344175


In [40]:
daily_pnl.group_by(['date']).agg([
    pl.col('Weighted PnL').sum().alias('Total_PnL')
]).sort('date')

date,Total_PnL
date,f64
2025-05-12,-10.104278
2025-05-13,2.584278
2025-05-14,0.0


In [11]:
import requests
import polars as pl 
from datetime import datetime, timedelta
# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol=AUAD.LON&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
r = requests.get(url)
data = r.json()
#pl.read_json(data)
# Convert JSON to Polars DataFrame
df = pl.DataFrame([
    {'date': datetime.strptime(date, '%Y-%m-%d').date(), **{k.split('. ')[1]: float(v) for k, v in values.items()}}
    for date, values in data['Time Series (Daily)'].items()
])

#daily_pnl_etf= calculate_daily_pnl(df, 1000, (datetime.now()-timedelta(days=5)).date(), datetime.now().date())

ytd_start = datetime(2025, 5, 12).date()
current_date = datetime.now().date()

period_data = df.filter(
        (pl.col('date') >= ytd_start) & 
        (pl.col('date') <= current_date)
    )
period_data

date,open,high,low,close,adjusted close,volume,dividend amount,split coefficient
date,f64,f64,f64,f64,f64,f64,f64,f64
2025-06-05,1880.5,1882.75,1877.5,1882.75,1882.75,45.0,0.0,1.0
2025-06-04,1883.0,1883.0,1874.5,1875.5,1875.5,1031.0,0.0,1.0
2025-06-03,1855.5,1863.25,1855.5,1863.25,1863.25,5332.0,0.0,1.0
2025-06-02,1853.5,1856.25,1848.0,1856.25,1856.25,68150.0,0.0,1.0
2025-05-30,1845.5,1845.5,1844.0,1844.25,1844.25,8.0,0.0,1.0
…,…,…,…,…,…,…,…,…
2025-05-16,1848.5,1851.0,1844.0,1846.25,1846.25,904.0,0.0,1.0
2025-05-15,1847.0,1850.5,1846.0,1850.5,1850.5,18.0,0.0,1.0
2025-05-14,1843.23,1843.252,1833.75,1833.75,1833.75,1381.0,0.0,1.0


In [47]:
round(((1853-1835)/1835 ) *100)


1